In [1]:
import torch
from torchvision import datasets, transforms
from sklearn.metrics import mean_absolute_error
from torch.utils.data import DataLoader    
import numpy as np
from sklearn.ensemble import RandomForestClassifier

# Loading the MNIST dataset

In [2]:
transform = transforms.ToTensor()

mnist_train = datasets.MNIST(root='../datasets/data', train=True, transform=transform)
mnist_test = datasets.MNIST(root='../datasets/data', train=False, transform=transform)

# Creating X and y variables

In [3]:
X_train = np.array([img.numpy().flatten() for img, label in mnist_train])  # flatten 28x28 → 784
y_train = np.array([label for img, label in mnist_train])
X_test = np.array([img.numpy().flatten() for img, label in mnist_test])
y_test = np.array([label for img, label in mnist_test])

# Tuning Parameters

In [4]:
n_estimators = [i for i in range(50, 350, 50)]
print(n_estimators)
best_n = []

for i in n_estimators:
    testf = RandomForestClassifier(random_state=42, n_estimators=i)
    testf.fit(X_train, y_train)
    pred = testf.predict(X_test)
    mae =  mean_absolute_error(y_test, pred)
    best_n.append((mae, i))
    print("Estimator: {} / Loss: {}".format(i, mae))

best_n = min(best_n)[1]

    


[50, 100, 150, 200, 250, 300]
Estimator: 50 / Loss: 0.1292
Estimator: 100 / Loss: 0.1176
Estimator: 150 / Loss: 0.1181
Estimator: 200 / Loss: 0.1162
Estimator: 250 / Loss: 0.1144
Estimator: 300 / Loss: 0.1129


# Building Model

In [5]:

clf = RandomForestClassifier(random_state=42, n_estimators=best_n)
clf.fit(X_train, y_train)
accuracy = clf.score(X_test, y_test)
print('Accuracy: {:.2f}%'.format(accuracy * 100))

Accuracy: 97.15%


# Including the DIDA dataset for historical support

In [6]:
from torchvision.datasets import ImageFolder  
from torch.utils.data import ConcatDataset
from torch.utils.data import random_split 
from sklearn.model_selection import GridSearchCV  

mnist_transform = transforms.Compose([
    transforms.Grayscale(num_output_channels=1),
    transforms.Resize((28, 28)),
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,))
])

dida_transform = transforms.Compose([
    transforms.Grayscale(num_output_channels=1),
    transforms.Resize((28, 28)),
    transforms.ToTensor(),
    transforms.Lambda(lambda x: 1 - x),  # invert only DIDA
    transforms.Normalize((0.1307,), (0.3081,))
])

dida_data = ImageFolder(root='../datasets/dida/70000', transform=dida_transform)
mnist_data = datasets.MNIST(root='../datasets/data', train=True, download=True, transform=mnist_transform)
mnist_test = datasets.MNIST(root='../datasets/data', train=False, download=True, transform=mnist_transform)
generator = torch.Generator().manual_seed(42)

dida_train, dida_test = random_split(dida_data, [0.8, 0.2], generator=generator)
train_data = ConcatDataset([mnist_data, dida_train])
test_data = ConcatDataset([mnist_test, dida_test])

X_train = np.array([img.numpy().flatten() for img, label in train_data])  # flatten 28x28 → 784
y_train = np.array([label for img, label in train_data])
X_test = np.array([img.numpy().flatten() for img, label in test_data])
y_test = np.array([label for img, label in test_data])

idx = np.random.permutation(len(X_train))
X_train, y_train = X_train[idx], y_train[idx]


In [7]:
n_estimators = [i for i in range(100, 400, 100)]
best_n = []
for i in n_estimators:
    model = RandomForestClassifier(random_state=42, n_estimators=i)
    model.fit(X_train, y_train)
    pred = model.predict(X_test)
    mae = mean_absolute_error(y_test, pred)
    print("Estimator: {} \n Loss: {}".format(i,mae))
    best_n.append((mae, i))
    
best_n=min(best_n)[1]

Estimator: 100 
 Loss: 0.31695833333333334
Estimator: 200 
 Loss: 0.30491666666666667
Estimator: 300 
 Loss: 0.30141666666666667


In [8]:
clf = RandomForestClassifier(random_state=42, n_estimators=best_n)
clf.fit(X_train, y_train)
accuracy = clf.score(X_test, y_test)
print('Accuracy: {:.2f}%'.format(accuracy * 100))

Accuracy: 91.92%


In [9]:
from sklearn.metrics import confusion_matrix

confusion_matrix(y_test, pred)

array([[2253,    7,   10,   10,   15,    9,   27,    9,    8,   12],
       [   9, 2424,   15,   12,   14,    7,   29,   25,    9,   11],
       [  14,    4, 2154,   24,   24,   20,   18,   27,   60,   24],
       [  20,    1,   42, 2302,   16,   30,    1,   14,   33,   18],
       [  11,   27,   24,   14, 2207,   10,   28,   32,   30,   38],
       [  11,    1,   10,   55,   14, 2174,   20,    9,   14,   11],
       [  30,   16,    5,    3,    5,   37, 2205,    6,   15,    0],
       [   5,   27,   47,   17,   16,    7,   16, 2232,   22,   55],
       [  33,    3,   32,   47,   34,   59,   66,   27, 2041,   21],
       [  18,   15,   33,   65,   51,   19,    5,   72,   24, 2068]])